In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, when, trim, lower, count, avg, desc, lit, round

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("schema_origen", "silver")
dbutils.widgets.text("schema_destino", "golden")
dbutils.widgets.text("catalog", "catalog_pry_final")
dbutils.widgets.text("schema_bronze", "bronze")

In [0]:
schema_origen = dbutils.widgets.get("schema_origen")
schema_bronze = dbutils.widgets.get("schema_bronze")
schema_destino = dbutils.widgets.get("schema_destino")
catalog = dbutils.widgets.get("catalog")

In [0]:
dm_user_profiling = spark.table(f"{catalog}.{schema_bronze}.ordenes").groupBy("user_id").agg(
    count("orden_id").alias("total_ordenes"),
    round(avg("dias_desde_orden"),4).alias("prom_dias_entre_ordenes")
).orderBy(desc("total_ordenes"))

In [0]:
df_dim_products = spark.table(f"{catalog}.{schema_origen}.dim_productos")
df_ordenes_prior = spark.table(f"{catalog}.{schema_bronze}.ordenes_prior")

In [0]:
# Data Mart 2: Rendimiento de Categorías/Departamentos
dm_department_performance = df_ordenes_prior \
    .join(df_dim_products, "producto_id", "inner") \
    .groupBy("departamento_nombre") \
    .agg(
        count("orden_id").alias("total_productos_vendidos")
    ).orderBy(desc("total_productos_vendidos"))

# Data Mart 3: Top 10 Productos Más Vendidos
dm_top_products = df_ordenes_prior\
    .groupBy("producto_id") \
    .agg(count("orden_id").alias("total_vendido")) \
    .join(df_dim_products, "producto_id", "inner") \
    .select("producto_nombre", "departamento_nombre", "total_vendido") \
    .orderBy(desc("total_vendido")) \
    .limit(10)

In [0]:
dm_department_performance.write.mode("overwrite").insertInto(f"{catalog}.{schema_destino}.dm_department_performance")
dm_top_products.write.mode("overwrite").insertInto(f"{catalog}.{schema_destino}.dm_top_products")
dm_user_profiling.write.mode("overwrite").insertInto(f"{catalog}.{schema_destino}.user_profiling")